<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/LangChain_101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LangChain 101: One Interface, and an Agent in Five Lines

You have built every core LLM pattern in this course *directly*: API calls, RAG, memory, routing, and a full agent loop — first by hand, then with each provider's native tool calling. So you are in the rare position of knowing exactly what a framework would have to automate to earn its place.

This notebook is the hands-on for **LangChain 1.x**, the framework half of the course's production stack. Two pieces, both small:

1. **`init_chat_model()`** — one function that turns a `"provider:model"` string into a chat model, so switching providers is a string change (the same idea as the course toolkit's `PROVIDER` switch, industrialized).
2. **`create_agent()`** — the agents-lesson loop as one call: model + tools in, a runnable agent out, with the loop, schema translation, and dispatch handled for you — running on the **LangGraph** runtime that the production tutor uses.

📎 *Nothing here replaces what you built — it packages it. The point of this lesson is to let you see through the abstraction, not to hide behind it.*

## 🧭 What You'll Learn

- `init_chat_model("provider:model")` — one interface over Gemini, OpenAI, and Anthropic, with `.invoke()` and `.stream()`
- Messages in, `AIMessage` out: LangChain's provider-neutral message objects
- Declaring a tool with the `@tool` decorator — the docstring becomes the tool description
- `create_agent()`: the whole tool-calling loop from the agents lesson as a single call
- Reading an agent's message trace, and streaming its steps as they happen
- What the framework bought you — and the doors (persistence, human-in-the-loop) it opens ⏭️

## 1. Setup: Environment, Keys, and Pins

Pinned as of August 2026 — LangChain 1.x. One provider key is enough; the `PROVIDER` dropdown decides which.

In [1]:
# ============================================================
# ⚙️ Setup — environment, dependencies, API keys, provider
# ============================================================
import os
import sys

IN_COLAB = "google.colab" in sys.modules

# Pick your model provider (dropdown in Colab; edit the values locally)
PROVIDER = "gemini"  # @param ["gemini", "openai", "anthropic"]

# init_chat_model() takes one "provider:model" string — the whole provider
# switch lives in this little table (course-standard models, August 2026):
_MODEL_FOR = {
    "gemini": "google_genai:gemini-3.7-flash",
    "openai": "openai:gpt-5.6-luna",
    "anthropic": "anthropic:claude-sonnet-5",
}
MODEL = _MODEL_FOR[PROVIDER]

_KEY_FOR = {"gemini": "GOOGLE_API_KEY", "openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY"}
REQUIRED_KEYS = [_KEY_FOR[PROVIDER]]

if IN_COLAB:
    import importlib
    import site
    import subprocess

    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "-U",
            "tai-aitutor==0.0.3",
            "langchain==1.3.14",
            "langgraph==1.2.10",
            "langchain-google-genai==4.3.2",
            "langchain-openai==1.4.1",
            "langchain-anthropic==1.5.4",
        ],
        check=True,
    )
    importlib.reload(site)  # make newly installed packages importable without a runtime restart

if not IN_COLAB:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
    # Locally: pip install the same pinned packages once; keys live in a .env file.

from tai_aitutor import setup_notebook

# The standard course key loader — Colab Secrets or a local .env, same code path.
setup_notebook(required_keys=REQUIRED_KEYS)

print(f"✅ Setup complete — {'Colab' if IN_COLAB else 'local'} | model: {MODEL}")

✅ Setup complete — local | model: google_genai:gemini-3.7-flash


## 2. `init_chat_model`: the Provider Switch as a String

In the agents lesson you wrote three different native SDK dialects. LangChain's first pitch is collapsing that: **one constructor, one calling convention**, whatever the provider.

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model(MODEL)  # e.g. "google_genai:gemini-3.7-flash"

reply = model.invoke("In one sentence, what does a reranker do in a RAG pipeline?")

print(type(reply).__name__)  # AIMessage — the provider-neutral reply object
print(reply.content)

AIMessage
[{'type': 'text', 'text': 'In a RAG pipeline, a reranker re-scores and reorders an initial set of retrieved documents to ensure the language model receives the most accurate and relevant context.', 'extras': {'signature': 'ErIMCq8MARFNMg/gUYuuKn8bHnwGCHuLO8RIaT3rlz/bFW5v4qIjINUBtR27hcPrnTC0uc8mQwAqgE/bvmDafmcWRP7/cEF+SRvWYMGWJyHaf8Fu0Tob/dpxaOU0t5mwIR6Q5/Ig6hKIxvh4HsvvqKbbd4dy3i0f7qgktA2g3Ve1GAgrXwnbWMrw7agrH93A8W2Iq/yXtWzOyZI6gt9y2ttLPIDNz15HDhXWWL7cWmwj9CK3ra0L07gaO2SvPek2YzuOuUs2VH0bJG5XKujpUqh2YbklYU+CpIYyLdd079H+02AE4O68ED25rDZ5UXg7tOk0gvhaIh+aFLH+2Ws+TEtBT4GJhQXf0/vCU2y2oVAbzskR4t+82/aney9mufyiDjZdIItOliM4P4BbtpxGJzi5CxTln3TnNmEZ6XlsrKuIPFlkwdCHWDeZ1D8zMM9JBHY+1gR9ac9+4T/uRZvpHo786BR5shMJQ1OqSwGozeLCFj+fT9U8YokWhXwZ3362zE1O64KgSorQ4YKIfFqq29c4QfokddSNOzUGBqQIBgsZNF6o23nY2qPJhURGaWW34UfB+QpjEyt6VCLDpsI4nNv6HK/2rhsAp1geqevWTqCUJAFpuyNAJqIT+vTigXjYWoqb0RwvivNABGVXbN5KBUeD09MTU9tFNaCmGRvy7Qq7DdSrV2esu/42+PlSQqLJNNU/r/7MgWH2obos+uf6s0qX48xP/NAsK9QM962TK1AeDqqwRRotJwqDsuDb/1p

In [3]:
# Conversations are lists of messages — same shape as every provider API,
# but provider-neutral. Roles can be plain dicts or message classes.
messages = [
    {"role": "system", "content": "You are a terse RAG course assistant. One sentence per answer."},
    {"role": "user", "content": "Why do we chunk documents before embedding them?"},
]

print(model.invoke(messages).content)

[{'type': 'text', 'text': "We chunk documents to fit within the embedding model's token limits and to improve retrieval accuracy by isolating distinct semantic concepts.", 'extras': {'signature': 'EvMMCvAMARFNMg901B4TMh1d8PPOS1CieUx2YH8AVM0/8tVImXi4bHJIRvHkHffKEgnQ1u2/w8vbVi7y70R8rGYihx8zSZolZzlWkQmFpAArQ2lbilxU0Grk0VR75AHcGtUoqabIerzeq6JxDoGFM4moCz3GkHfExOiug71xTHijcdDlKfHgn7P77kAkGgzTRxDfAE73KwJYnxPS5L9+QM4gqeggvN6RLUlKh6aZAwDbNfLVA8m5LmeW8FtN7KvPV750fw1snKAnzCkhZN7sRlXOCg/ufUcm2Et+e6LUXyEoPs8dnLaCyv2+fu30UcrW3EK/TxFOX1x1Q3+W62agNap+Iyt8BsOzf6rsKFycjvgBTW1PLsMSsnoI9oBcp/AdM5hWnlOh6PjSHJ9uobhCTWVmmOpmkMMewXI0D/jDRUYOi/DMlnYJtrACEkqqNBKNCjtijD+7xaF2khFrEmg6paFrsRTd1nGAFCHb+GC9hg+JKSBW1pb4R0VqUPWNkLOb8xbT9HHMAAzOEgnaY4rk5oveQkXp8Z6FXO/yUiEjFTgk+0k+8Nc38YqVMpeOVSQMVsqGNRyOdTdxCXWrjKqWAhAbnFwMjQ7cd6OZtfdv6z7RqHQmnOHk7H/ufiXoNN46RLBeblN3jeybpAxQvhj//n+uIYDZtRHs8RxPcQtZRYYy2ULuJXrFHpzzFneoAw49VUsosuSaQxTA1diB9hhXgvQm2puSl0ZIWVKDmnwQSnQuDXetbeEmU0qhRUw0yrLFW+VMECfobsoKzovEC3lqSwFzl3Lgwe3ym13

In [4]:
# Streaming is a method, not a protocol change:
for chunk in model.stream("Explain BM25 to a five-year-old in two sentences."):
    print(chunk.content, end="", flush=True)
print()

[{'type': 'text', 'text': 'Imagine a smart robot that helps you find the best book by counting how many times your favorite word shows up in it. It', 'index': 0}][{'type': 'text', 'text': ' gives extra gold stars to rare, special words so you get the most helpful story instead of just the longest book!', 'index': 0}][{'type': 'text', 'text': '', 'extras': {'signature': 'EuYSCuMSARFNMg+9WWkmE7syIhgl7+2GRgtQboHppOQUH3UXDG6JBGmwZV9SXo3ChJJLD/HyrblW8SknGu7M8/LDLjQxmkje9dhewpzYsV4YAAc9eEebf9V6ZQEEM7E7ZCbUbDWNpefNtJ8MxVcXOLurQVCCnWDFe7TL7VYjDW+IdO82HZkb1zBibAa7lVb2h5Y9A2QJAcG0EVuiw3Ue3anPGy/gh1ePdHQmQMhyPof2eMzVVJvx6RC24o3cQoc7bZOeINRalSTgHtG5n8ACXPLegPveHpJiNKFgX9bRTrQTXUa7EXWispIxs4awTrYMcEV03TrDZEVsxUVUcpSIfog8GqDfeDfpZC0od8j6i3s1G6rb2GwXFtKB83vFQqGKjxTvT+Ox8HcZFuJnWa9GyrwzuaKsbEO37H8jmDewXwzbkDkyb5/17x0QGTLfy6hBQcPoV/aaEbImwkeNoEYNUxaE61wogolnpCGsY+lCphQp8W2uYdy/Wt3WxuqxDjP0l1FwHP87+AfxZt6yd8TfFqBfAQYHY1ZXV+xvFimJZKOKB1va8cOIefKc8LzUNWSjLIbsCq0NE7ZluNSQ1WvGVsfgREIg+fd3lqFZ1YkdDx8o7e9+UK0

**What just happened?** Three calls, zero provider-specific code. Change `PROVIDER` in Setup and re-run — same cells, different model underneath. The `AIMessage` object also carries `usage_metadata` (token counts) and, when tools are involved, structured `tool_calls` — the same information you handled by hand in the three native dialects, normalized into one shape.

## 3. A Custom Tool: `@tool`

A LangChain tool is a plain function with the `@tool` decorator. The function name becomes the tool name, the **docstring becomes the tool description**, and the type hints become the JSON Schema — the exact schema you wrote by hand in the agents lesson, now generated for you. (Gemini's SDK did the same trick natively; LangChain does it for every provider.)

In [5]:
from langchain_core.tools import tool

# 📎 The allowlisted calculator from the agents lesson, in compact form —
# same rule as always: never eval() model-written strings.
import ast
import operator

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}


def _safe_eval(node):
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("unsupported expression")


@tool
def calculate(expression: str) -> str:
    """Evaluate an arithmetic expression like '337.2 / 3' (numbers and + - * / ** only)."""
    return str(_safe_eval(ast.parse(expression.strip(), mode="eval")))


print(calculate.name, "|", calculate.description)
print(calculate.invoke({"expression": "21 * 2"}))

calculate | Evaluate an arithmetic expression like '337.2 / 3' (numbers and + - * / ** only).
42


## 4. `create_agent`: the Loop You Built, as One Call

Recall what the agents-lesson loop needed: a system prompt, tool schemas, a dispatch table, observation passing, and a turn limit. `create_agent()` is all of that in one call — and the object it returns is a compiled **LangGraph** graph, which is what makes it production-grade (persistence, streaming, human-in-the-loop are runtime features, not rewrites).

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    model,
    tools=[calculate],
    system_prompt=(
        "You are a precise assistant for a RAG engineering course. "
        "Use the calculate tool for any arithmetic instead of doing it yourself."
    ),
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "A corpus has 3,203 documents averaging 2,691 tokens. Roughly how many million tokens is that?"}]}
)

print(result["messages"][-1].content)

[{'type': 'text', 'text': 'That is roughly **8.62 million tokens** (8,619,273 tokens).', 'extras': {'signature': 'EqYBCqMBARFNMg9Xyu8RIf5zumN4YkSiMJUtU8QO6TzT/x4chOOAFLK9MBgEGCrFx+yY9VZtOioyIKUws+Shot3CQVb+9NPI8qt31FopQNvEBgAPx21Hkjqk9eM+jTKVYyqfELMR8wT9Eh4O2yQNvwcITt9+DlXuczjpky7srYKz56bcqJdoQHA/eJGy2v61u4wf9wwrIIIduj4mKl6Ea+25ERYFlZdh6A=='}}]


In [7]:
# The full trace is in the returned state — every message the loop produced:
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

A corpus has 3,203 documents averaging 2,691 tokens. Roughly how many million tokens is that?
================================== Ai Message ==================================

[]
Tool Calls:
  calculate (call_795987)
 Call ID: call_795987
  Args:
    expression: (3203 * 2691) / 1000000
================================= Tool Message =================================
Name: calculate

8.619273
================================== Ai Message ==================================

[{'type': 'text', 'text': 'That is roughly **8.62 million tokens** (8,619,273 tokens).', 'extras': {'signature': 'EqYBCqMBARFNMg9Xyu8RIf5zumN4YkSiMJUtU8QO6TzT/x4chOOAFLK9MBgEGCrFx+yY9VZtOioyIKUws+Shot3CQVb+9NPI8qt31FopQNvEBgAPx21Hkjqk9eM+jTKVYyqfELMR8wT9Eh4O2yQNvwcITt9+DlXuczjpky7srYKz56bcqJdoQHA/eJGy2v61u4wf9wwrIIIduj4mKl6Ea+25ERYFlZdh6A=='}}]


**What just happened?** Read the trace bottom-up against the agents lesson: a human message, an AI message *with a `tool_calls` entry* (the structured request you used to regex out of text), a **ToolMessage** carrying the calculator's result (your `Observation:`), and a final AI answer. Identical loop, zero loop code — and no `eval()` anywhere: the dispatch is the framework's, but the tool body is still yours, allowlist and all.

In [8]:
# Agents stream too — one event per loop step, as it happens:
for event in agent.stream(
    {"messages": [{"role": "user", "content": "What is (84 - 7) * 12? Then say which tool you used."}]},
    stream_mode="updates",
):
    for node_name, update in event.items():
        latest = update["messages"][-1]
        print(f"[{node_name}] {type(latest).__name__}: {str(latest.content)[:120]}")

[model] AIMessage: []
[tools] ToolMessage: 924
[model] AIMessage: [{'type': 'text', 'text': '(84 - 7) * 12 = 924.\n\nI used the **`calculate`** tool to evaluate the expression.', 'extras


**What just happened?** `stream_mode="updates"` yields one event per **node** of the underlying graph — you can watch the model node and the tool node take turns, which is exactly your Thought → Action → Observation cycle wearing framework clothes. (Token-level streaming of the final answer is `stream_mode="messages"` — try it.)

## 5. So… Do You Need It?

Now you can answer the framework question from experience, not marketing:

- **What it automated:** the loop, the per-provider tool-schema dialects, message normalization, streaming plumbing.
- **What it did not change:** your tools, your prompts, your retrieval pipeline, your judgment about what the agent should do — and your responsibility for safety (the allowlisted calculator matters exactly as much as it did last lesson).
- **What it unlocks:** because `create_agent()` returns a LangGraph graph, per-thread **persistence**, **streamed events**, and **human-in-the-loop** pauses are configuration, not rewrites. That is why the production tutor in Section 13 runs `create_agent` on LangGraph.

For a small pipeline, the direct code from Part 1 remains a perfectly good answer — it is fewer dependencies and nothing is hidden. Reach for the framework when you need what its runtime gives you.

## 🧪 Your Turn

- Flip `PROVIDER` to another vendor and re-run everything — count the lines you had to change.
- Add a second tool (port `search()` from the agents lesson with `@tool`) and ask a question needing both.
- Ask something needing *no* tool — check the trace: does the agent call the calculator anyway?
- Try `stream_mode="messages"` on the agent for token-level output.

## 🔑 Key Takeaways

- **`init_chat_model("provider:model")`** makes the provider a string; `.invoke()` / `.stream()` and `AIMessage` are the same everywhere.
- **`@tool` turns a function into a schema** — name from the function, description from the docstring, schema from type hints. Write docstrings like the model is reading them, because it is.
- **`create_agent(model, tools, system_prompt=…)`** is the agents-lesson loop as one call, returning a compiled LangGraph graph; `.invoke()` takes and returns `{"messages": [...]}` state.
- The trace (`result["messages"]`, `.pretty_print()`) shows the same Thought/Action/Observation structure you built by hand — always read it before trusting an agent.
- Frameworks automate the loop, **not** the safety: tool bodies still treat model arguments as untrusted input.
- ⏭️ *Next: LangGraph itself — the state graphs under `create_agent`, with checkpointed memory, streaming modes, and `interrupt()` for human approval. Section 13's production tutor is exactly that stack.*